### portmy database: stocks table
### stock database: buy, price tables

In [1]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()
conmy = engine.connect()
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.float_format','{:,.2f}'.format)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"

today = date.today()
today

datetime.date(2023, 1, 14)

### Process after last business day, yesterday must be last business day

In [2]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-14
yesterday: 2023-01-13 00:00:00


In [3]:
yesterday = yesterday.date()
week_ago = yesterday -timedelta(days = 7)
print(f'a week ago: {week_ago}')
print(f'yesterday: {yesterday}')

a week ago: 2023-01-06
yesterday: 2023-01-13


### Restart & Run All Cells

### This process affects only already in port stocks. To highlight price changes in percent.

In [4]:
cols = 'name price_w price_d percent perform mrkt '.split()

format_dict = {
    'qty':'{:,}',    
    'price':'{:.2f}','price_d':'{:.2f}','price_w':'{:.2f}',
    'max_price':'{:.2f}','min_price':'{:.2f}',
    'maxp':'{:.2f}','minp':'{:.2f}','opnp':'{:.2f}',    
    'pct':'{:,.2f}%','percent':'{:,.2f}%',
    
    'pe':'{:.2f}','pbv':'{:.2f}',
    'paid_up':'{:,.2f}','market_cap':'{:,.2f}',
    'daily_volume':'{:,.2f}','beta':'{:,.2f}', 
    'dly_vol':'{:,.2f}', 
}

In [5]:
sql = """
SELECT name, date, price 
FROM buy 
ORDER BY date DESC 
LIMIT 1
"""
df_buy = pd.read_sql(sql, const)
df_buy.style.format(format_dict)

,name,date,price
0,BANPU,2023-01-06,12.30


In [6]:
names = df_buy['name']
name = names.to_string(index=False)

sql = '''
SELECT * FROM stocks WHERE name = '%s'
'''
sql = sql % name
print(sql)

df_stocks = pd.read_sql(sql, conmy)
df_stocks.style.format(format_dict) 


SELECT * FROM stocks WHERE name = 'BANPU'



,id,name,market,price,max_price,min_price,pe,pbv,paid_up,market_cap,daily_volume,beta,ticker_id,created_at,updated_at
0,47,BANPU,SET50 / SETCLMV / SETTHSI,12.50,15.00,10.50,2.39,0.95,"8,454.16","105,677.02","2,029.11",0.76,47,2017-07-23 07:24:25.883582,2023-01-14 04:15:01.317728


In [7]:
sql = '''
SELECT * FROM price WHERE name = "%s" ORDER BY date DESC LIMIT 5
'''
sql = sql % name
print(sql)

df_price = pd.read_sql(sql, const)
df_price.style.format(format_dict)


SELECT * FROM price WHERE name = "BANPU" ORDER BY date DESC LIMIT 5



,name,date,price,maxp,minp,qty,opnp
0,BANPU,2023-01-13,12.50,12.50,12.20,"107,438,620",12.30
1,BANPU,2023-01-12,12.30,12.40,12.10,"83,041,084",12.20
2,BANPU,2023-01-11,12.40,12.50,12.20,"146,184,437",12.50
3,BANPU,2023-01-10,12.50,12.60,12.30,"104,568,150",12.50
4,BANPU,2023-01-09,12.50,12.50,12.20,"151,235,816",12.30


In [8]:
sql = '''
SELECT name
FROM buy 
WHERE active = 1
ORDER BY name'''
df_price = pd.read_sql(sql, const)

names = df_price.name.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))

sql = """
SELECT name, price AS price_d 
FROM price 
WHERE date = '%s' AND name IN (%s)
ORDER BY name, date"""
sql = sql % (yesterday, in_p)
print(sql)

df_today = pd.read_sql(sql, const)
df_today.shape[0]


SELECT name, price AS price_d 
FROM price 
WHERE date = '2023-01-13' AND name IN ('ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART')
ORDER BY name, date


25

In [9]:
sql = """
SELECT name, price AS price_w
FROM price 
WHERE date = '%s' AND name IN (%s) 
ORDER BY name"""
sql = sql % (week_ago, in_p)
print(sql)

df_wkago = pd.read_sql(sql, const)
df_wkago.shape[0]


SELECT name, price AS price_w
FROM price 
WHERE date = '2023-01-06' AND name IN ('ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART') 
ORDER BY name


25

In [10]:
trend = pd.merge(df_today, df_wkago, on='name',how='inner')
trend.shape

(25, 3)

In [11]:
def performance(vals):
    price_d, price_w = vals
    if (price_d > price_w):
        return 'Better'
    elif (price_d < price_w):
        return 'Worse'
    else:
        return 'No Change'

In [12]:
trend['percent'] = (trend.price_d-trend.price_w)/trend.price_w * 100

In [13]:
trend

,name,price_d,price_w,percent
0,ASP,3.06,2.98,2.68
1,BANPU,12.50,12.30,1.63
2,BCH,21.50,21.90,-1.83
3,CPNREIT,19.50,19.50,0.00
4,DCC,2.82,2.84,-0.70
5,DIF,13.50,13.20,2.27
6,GVREIT,9.00,9.00,0.00
7,IVL,41.75,37.75,10.60
8,JASIF,8.20,8.15,0.61
9,KCE,52.00,48.00,8.33


In [14]:
trend["perform"] = trend[["price_d","price_w"]].apply(performance, axis=1)

In [15]:
trend.sort_values(by=['percent'],ascending=[True]).style.format(format_dict)

,name,price_d,price_w,percent,perform
2,BCH,21.50,21.90,-1.83%,Worse
4,DCC,2.82,2.84,-0.70%,Worse
3,CPNREIT,19.50,19.50,0.00%,No Change
6,GVREIT,9.00,9.00,0.00%,No Change
10,MAKRO,42.50,42.50,0.00%,No Change
18,SENA,3.92,3.92,0.00%,No Change
23,WHAIR,7.50,7.50,0.00%,No Change
17,SCCC,158.50,158.50,0.00%,No Change
20,SYNEX,17.10,17.00,0.59%,Better
8,JASIF,8.20,8.15,0.61%,Better


### Filter only performance = "Worse"

In [16]:
mask = trend.perform == 'Worse'
trend[mask]

,name,price_d,price_w,percent,perform
2,BCH,21.50,21.90,-1.83,Worse
4,DCC,2.82,2.84,-0.70,Worse


In [17]:
trend.perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,68.00%
No Change,24.00%
Worse,8.00%


In [18]:
sql = '''
SELECT name, max_price AS max, min_price AS min, pe, pbv, daily_volume AS volume, beta, market
FROM stocks
'''
my_stocks = pd.read_sql(sql, conmy)
my_stocks.shape

(253, 8)

In [19]:
filters = [
   (my_stocks.market.str.contains('SET50')),
   (my_stocks.market.str.contains('SET100')),
   (my_stocks.market.str.contains('mai'))]
values = ['SET50','SET100','mai']
my_stocks["mrkt"] = np.select(filters, values, default='SET999')

In [20]:
trend2 = pd.merge(trend, my_stocks, on='name', how='inner')
trend2.sort_values(['percent'],ascending=[True]).shape

(25, 13)

In [21]:
set50 = trend2.mrkt.str.contains('SET50') 
flt_set50 = trend2[set50]
flt_set50[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt
16,SCC,350.00,355.00,1.43%,Better,SET50
1,BANPU,12.30,12.50,1.63%,Better,SET50
14,PTTGC,46.25,50.00,8.11%,Better,SET50
7,IVL,37.75,41.75,10.60%,Better,SET50


In [22]:
flt_set50.perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,100.00%


In [23]:
set100 = trend2.mrkt.str.contains('SET100')
flt_set100 = trend2[set100]
flt_set100[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt
2,BCH,21.90,21.50,-1.83%,Worse,SET100
19,STA,21.10,21.30,0.95%,Better,SET100
13,ORI,11.60,11.90,2.59%,Better,SET100
15,RCL,30.50,31.50,3.28%,Better,SET100
9,KCE,48.00,52.00,8.33%,Better,SET100


In [24]:
flt_set100[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,80.00%
Worse,20.00%


In [25]:
set999 = trend2.mrkt.str.contains('SET999')
flt_set999 = trend2[set999]
flt_set999[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt
4,DCC,2.84,2.82,-0.70%,Worse,SET999
3,CPNREIT,19.50,19.50,0.00%,No Change,SET999
6,GVREIT,9.00,9.00,0.00%,No Change,SET999
10,MAKRO,42.50,42.50,0.00%,No Change,SET999
17,SCCC,158.50,158.50,0.00%,No Change,SET999
18,SENA,3.92,3.92,0.00%,No Change,SET999
23,WHAIR,7.50,7.50,0.00%,No Change,SET999
20,SYNEX,17.00,17.10,0.59%,Better,SET999
8,JASIF,8.15,8.20,0.61%,Better,SET999
24,WHART,10.70,10.90,1.87%,Better,SET999


In [26]:
flt_set999[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,56.25%
No Change,37.50%
Worse,6.25%


In [27]:
setmai = trend2.mrkt.str.contains('mai')
flt_setmai = trend2[setmai]
flt_setmai[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt


In [28]:
flt_setmai[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
